# Perform inference with CTlearn models
For this notebook, one would need to download the HDF5 data files
from the CTAO opendata folder (https://cloud.iaa.es/index.php/s/77d6MK4rGfanSKx) and the trained models of **train_ctlearn_model.ipynb**.


### Check paths, environments, CTAO test data, and trained models

In [ ]:
! pwd

In [ ]:
! conda list | grep ctapipe
! conda list | grep ctlearn
! conda list | grep dl1-data-handler 
! conda list | grep keras
! conda list | grep tensorflow

In [ ]:
TEST_DIR = "../testdata/ctao_opendata/test"
! du -h {TEST_DIR}/*
MODEL_DIR = "../my_outputs/"
! ls -l {MODEL_DIR}/*

### Explore the CTLearn prediction tools
Installing CTLearn provides several command-line tools for running and managing different parts of the workflow.
For model inference, we use the **ctlearn-predict-{mono,stereo}-model** command.
The *-h* option displays a short version of the help message, showing the main command-line options and their usage.
For a complete description of all available options, including the configuration parameters of the different components, use *--help-all*.
Although the command is executed from this Jupyter notebook for convenience and documentation, **ctlearn-predict-{mono,stereo}-model** is a command-line tool.
In a typical workflow, the prediction command would be executed directly from a terminal or submitted as part of a cluster job, for example through sbatch on an HPC system.

In [ ]:
! ctlearn-predict-mono-model -h
#! ctlearn-predict-mono-model --help-all
! ctlearn-predict-stereo-model -h
#! ctlearn-predict-stereo-model --help-all

## Monoscopic CNN-based model inference
We perform inference using the three previously trained CTLearn models from **train_ctlearn_model.ipynb**:
the primary particle type (classification), primary particle energy (regression), and arrival direction (regression) models.
The models are applied to both gamma-ray (signal) and proton (background) input data.
The three inference tasks are executed sequentially within a single tool invocation.
The prediction tool provides several options for selecting which data levels from the input file are copied to the output file.
By default, the goal is to progress to the next data level while keeping the output file as compact as possible.
However, retaining additional data levels can be useful for debugging and for inspecting individual events in more detail.
The resulting file contains the model predictions together with the selected input data levels and follows the CTAO reference data model.

**Important**: The same configuration settings used during training should be used for inference whenever applicable. In particular, the data preprocessing and data reader configuration must be consistent with those used to train the models.

In [ ]:
! mkdir ../my_predictions
for input_file in ["gamma-diffuse_with_images_10.dl2.h5", "proton_with_images_05.dl2.h5"]:
    output_file = input_file.replace(".dl2.h5", ".ctlearn.dl2.h5")
    ! ctlearn-predict-mono-model  \
        --input_url {TEST_DIR}/{input_file} \
        --output ../my_predictions/{output_file} \
        --config ../configs/dl1dh_example_config.json \
        --config ../configs/ctlearn_predict_model_example_config.json \
        --type_model {MODEL_DIR}/my_first_training_type/ctlearn_model.keras \
        --energy_model {MODEL_DIR}/my_first_training_energy/ctlearn_model.keras \
        --cameradirection_model {MODEL_DIR}/my_first_training_cameradirection/ctlearn_model.keras \
        --overwrite \
        --verbose

### Browsing through the HDF5 files via vitables
Use ViTables, a convenient GUI, to explore the CTLearn DL2 predictions. The file follows the reference implementation of the CTAO data model. 

In [ ]:
#! conda run -n vitables vitables ../my_predictions/*.h5

### add here exploration of DL2 data

In [ ]:
from matplotlib import pyplot as plt
from matplotlib.colors import LogNorm
import numpy as np
from ctapipe.io import TableLoader
from sklearn.metrics import roc_curve, auc

filenames = {
    "gamma": "../my_predictions/gamma-diffuse_with_images_10.ctlearn.dl2.h5",
    "proton": "../my_predictions/proton_with_images_05.ctlearn.dl2.h5",
    
}
predictions = {}
for particle_type, filename in filenames.items():
    loader = TableLoader(input_url=filename)
    telescope_events = loader.read_telescope_events(
        telescopes=["LST_LST_LSTCam"],
        dl1_images=False,
        dl1_parameters=False,
        dl1_muons=False,
        dl2=True,
        simulated=True,
        true_images=False,
        true_parameters=False,
        instrument=False,
        observation_info=False,
    )
    predictions[particle_type] = telescope_events[telescope_events["CTLearnRegressor_tel_is_valid"]]

# Extract predictions
gamma_predictions = np.asarray(
    predictions["gamma"]["CTLearnClassifier_tel_prediction"]
)
proton_predictions = np.asarray(
    predictions["proton"]["CTLearnClassifier_tel_prediction"]
)
# There is a small population of events in both classes that scores exactly 0.0.
# Maybe caused by by non optimal training process with very low statistics. We exclude them in the visulazation.
gamma_scores = gamma_predictions[gamma_predictions != 0.0]
proton_scores = proton_predictions[proton_predictions != 0.0]

# Store the energy reconstruced and true values
reco_energy = np.asarray(
    predictions["gamma"]["CTLearnRegressor_tel_energy"]
)
true_energy = np.asarray(
    predictions["gamma"]["true_energy"]
)

# Store the alt/az reconstruced and true values
reco_alt = np.asarray(
    predictions["gamma"]["CTLearnCameraReconstructor_tel_alt"]
)
true_alt = np.asarray(
    predictions["gamma"]["true_alt"]
)
reco_az = np.asarray(
    predictions["gamma"]["CTLearnCameraReconstructor_tel_az"]
)
true_az = np.asarray(
    predictions["gamma"]["true_az"]
)

# Astropy table can be displayed in the notebook
#predictions["gamma"].show_in_notebook(
#    backend="classic",
#    display_length=-1
#)

In [ ]:
# Prepare ROC data
# True labels: gamma = 1, proton = 0
y_true = np.concatenate([
    np.ones(len(gamma_scores)),
    np.zeros(len(proton_scores)),
])
y_score = np.concatenate([
    gamma_scores,
    proton_scores,
])
fpr, tpr, thresholds = roc_curve(y_true, y_score)
roc_auc = auc(fpr, tpr)

# Make the figure
fig, axes = plt.subplots(
    1, 2,
    figsize=(13, 5),
    constrained_layout=True,
)

# Left plot: gammaness distribution
bins = np.linspace(0, 1, 50)
axes[0].hist(
    gamma_scores,
    bins=bins,
    alpha=0.5,
    label="Signal (gamma)",
)
axes[0].hist(
    proton_scores,
    bins=bins,
    alpha=0.5,
    label="Background (proton)",
)
axes[0].set_xlabel("Classifier value (gammaness)")
axes[0].set_ylabel("Counts")
axes[0].set_title("Gammaness distribution")
axes[0].legend(loc="upper center")

# Right plot: ROC curve
axes[1].plot(
    fpr,
    tpr,
    lw=2,
    label=f"CTLearn classifier (AUC = {roc_auc:.3f})",
)
# Random-classifier reference
axes[1].plot(
    [0, 1],
    [0, 1],
    "k--",
    alpha=0.5,
    label="Random classifier",
)
axes[1].set_xlabel("False positive rate")
axes[1].set_ylabel("True positive rate")
axes[1].set_title("ROC curve")
axes[1].set_xlim(0, 1)
axes[1].set_ylim(0, 1)
axes[1].grid(alpha=0.3)
axes[1].legend(loc="lower right")
plt.show()

In [ ]:
# Create figure for the three migration matrices
fig, axes = plt.subplots(
    1, 3,
    figsize=(18, 5.5),
)

# Left plot: Energy migration
energy_bins = np.logspace(
    np.log10(1),
    np.log10(100),
    50,
)
hist_e, true_e_edges, reco_e_edges = np.histogram2d(
    true_energy,
    reco_energy,
    bins=[energy_bins, energy_bins],
)
hist_e_plot = np.ma.masked_where(
    hist_e == 0,
    hist_e,
)
mesh_e = axes[0].pcolormesh(
    true_e_edges,
    reco_e_edges,
    hist_e_plot.T,
    norm=LogNorm(
        vmin=1,
        vmax=hist_e.max(),
    ),
    cmap="viridis",
    shading="auto",
)
axes[0].plot(
    energy_bins,
    energy_bins,
    "k--",
    linewidth=1.5,
    label="Perfect reconstruction",
)
axes[0].set_xscale("log")
axes[0].set_yscale("log")
axes[0].set_xlabel("True energy [TeV]")
axes[0].set_ylabel("Reconstructed energy [TeV]")
axes[0].set_title("Energy migration")
axes[0].legend(loc="upper left")
cbar_e = fig.colorbar(
    mesh_e,
    ax=axes[0],
)
cbar_e.set_label("Number of events")

# Middle plot: Altitude migration [degrees]
alt_bins = np.linspace(
    min(true_alt.min(), reco_alt.min()),
    max(true_alt.max(), reco_alt.max()),
    50,
)
hist_alt, true_alt_edges, reco_alt_edges = np.histogram2d(
    true_alt,
    reco_alt,
    bins=[alt_bins, alt_bins],
)
hist_alt_plot = np.ma.masked_where(
    hist_alt == 0,
    hist_alt,
)
mesh_alt = axes[1].pcolormesh(
    true_alt_edges,
    reco_alt_edges,
    hist_alt_plot.T,
    norm=LogNorm(
        vmin=1,
        vmax=hist_alt.max(),
    ),
    cmap="viridis",
    shading="auto",
)
axes[1].plot(
    alt_bins,
    alt_bins,
    "k--",
    linewidth=1.5,
    label="Perfect reconstruction",
)
axes[1].set_xlabel("True altitude [deg]")
axes[1].set_ylabel("Reconstructed altitude [deg]")
axes[1].set_title("Altitude migration")
axes[1].legend(loc="upper left")
cbar_alt = fig.colorbar(
    mesh_alt,
    ax=axes[1],
)
cbar_alt.set_label("Number of events")

# Right plot: Azimuth migration centered on 0 degrees
# Wrap 0/360 degrees to [-180, 180)
true_az_wrapped = (true_az + 180) % 360 - 180
reco_az_wrapped = (reco_az + 180) % 360 - 180
# 0.5 degree bins
az_bins = np.linspace(
    -15,
    15,
    61,
)
hist_az, true_az_edges, reco_az_edges = np.histogram2d(
    true_az_wrapped,
    reco_az_wrapped,
    bins=[az_bins, az_bins],
)
hist_az_plot = np.ma.masked_where(
    hist_az == 0,
    hist_az,
)
mesh_az = axes[2].pcolormesh(
    true_az_edges,
    reco_az_edges,
    hist_az_plot.T,
    norm=LogNorm(
        vmin=1,
        vmax=hist_az.max(),
    ),
    cmap="viridis",
    shading="auto",
)
# Perfect reconstruction
axes[2].plot(
    [-15, 15],
    [-15, 15],
    "k--",
    linewidth=1.5,
    label="Perfect reconstruction",
)
axes[2].set_xlim(-15, 15)
axes[2].set_ylim(-15, 15)
axes[2].set_xlabel("True azimuth relative to 0° [deg]")
axes[2].set_ylabel("Reconstructed azimuth relative to 0° [deg]")
axes[2].set_title("Azimuth migration")
axes[2].set_xticks([-15, -10, -5, 0, 5, 10, 15])
axes[2].set_yticks([-15, -10, -5, 0, 5, 10, 15])
axes[2].legend(loc="upper left")
cbar_az = fig.colorbar(
    mesh_az,
    ax=axes[2],
)
cbar_az.set_label("Number of events")
plt.tight_layout()
plt.show()

## Stereoscopic CNN-based model inference
we perform inference with the previously trained stereoscopic energy model.
The telescope images are combined channel-wise and passed to the CNN, allowing the model to use information from multiple telescopes simultaneously.
As above, the same configuration used during training should be used for inference, in particular the stereoscopic image reader settings and telescope stacking options.
Since the stereoscopic input is larger than the monoscopic input, a reduced batch size may be required to account for the increased memory consumption.

In [ ]:
! mkdir ../my_predictions
for input_file in ["gamma-diffuse_with_images_10.dl2.h5", "proton_with_images_05.dl2.h5"]:
    output_file = input_file.replace(".dl2.h5", ".ctlearn.stereo.dl2.h5")
    ! ctlearn-predict-stereo-model  \
        --input_url {TEST_DIR}/{input_file} \
        --output ../my_predictions/{output_file} \
        --config ../configs/dl1dh_example_config.json \
        --config ../configs/ctlearn_predict_model_example_config.json \
        --DLImageReader.mode stereo \
        --DLImageReader.min_telescopes 2 \
        --StereoPredictCTLearnModel.stack_telescope_images True \
        --StereoPredictCTLearnModel.batch_size 16 \
        --energy_model {MODEL_DIR}/my_first_stereo_training_energy/ctlearn_model.keras \
        --overwrite \
        --verbose

## CNN-based model inference using waveform data
We perform inference with the previously trained mono energy model using waveform data.
The configuration must be consistent with the training setup.
An important configuration change is required to read waveform data: the *--MonoPredictCTLearnModel.dl1dh_reader_type* option must be set to the appropriate DL1 data reader component for waveform reading (**DLWaveformReader**) rather than the default image reader.

For this demonstration, we use the same data file as used during training, since suitable test data are currently not available. This is done only to demonstrate that the software workflow is functional. In a real analysis, using training data for testing is not valid practice.

In [ ]:
! mkdir ../my_predictions
input_file = "../testdata/gamma_prod5.r1.dl1.h5"
output_file = input_file.replace("testdata", "my_predictions").replace(".r1.dl1.h5", ".ctlearn.r1.dl1.dl2.h5")
! ctlearn-predict-mono-model  \
    --input_url {input_file} \
    --output {output_file} \
    --config ../configs/dl1dh_example_config.json \
    --config ../configs/ctlearn_predict_model_example_config.json \
    --MonoPredictCTLearnModel.dl1dh_reader_type DLWaveformReader \
    --MonoPredictCTLearnModel.batch_size 2 \
    --TableQualityQuery.quality_criteria [] \
    --energy_model {MODEL_DIR}/my_first_waveform_training_energy/ctlearn_model.keras \
    --overwrite \
    --verbose

## CNN-based model inference using LST-1 lstchain data
The following three cells demonstrate the prediction tool for LST-1 DL1 data in the lstchain format.
Since LST-1 data are private to the LST-1 collaboration and cannot be distributed, the first cell creates a mock dataset in the required format.
The second cell displays the tool help and its available options. The final cell performs inference on the mock data using the models trained in the previous notebook.
The tool predicts gammaness, energy, and arrival direction and writes the results to a ctapipe DL2 output file, together with the relevant LST-1 subarray, trigger, pointing, and DL1b parameter information.

**Note**: This prediction tool is specifically designed for the lstchain DL1 format and should not be used with other input data formats.

In [ ]:
from pathlib import Path
import numpy as np
from astropy.table import Column, Table
from ctapipe.io import write_table
from ctlearn.utils import get_lst1_subarray_description

output_path = Path("../testdata/mock_data_lst1.dl1.h5")
rng = np.random.default_rng(1234)
n_events = 25

subarray = get_lst1_subarray_description()
tel_id = 1
n_pixels = subarray.tel[tel_id].camera.geometry.n_pixels

obs_id = 1
event_ids = np.arange(1, n_events + 1, dtype=np.int64)

# Fake per-pixel data
image = rng.uniform(80, 150, size=(n_events, n_pixels)).astype(np.float32)
image_mask = rng.integers(0, 2, size=(n_events, n_pixels), dtype=bool)
peak_time = rng.normal(5.0, 0.5, size=(n_events, n_pixels)).astype(np.float32)

image_table = Table()
image_table["obs_id"] = np.full(n_events, obs_id, dtype=np.int64)
image_table["event_id"] = event_ids
image_table["tel_id"] = np.full(n_events, tel_id, dtype=np.int16)
image_table.add_column(
    Column(image, name="image", dtype=np.float32, shape=(n_pixels,))
)
image_table.add_column(
    Column(image_mask, name="image_mask", dtype=bool, shape=(n_pixels,))
)
image_table.add_column(
    Column(peak_time, name="peak_time", dtype=np.float32, shape=(n_pixels,))
)

# DL1 parameter columns required by LST1PredictionTool
parameter_table = Table()
parameter_table["obs_id"] = np.full(n_events, obs_id, dtype=np.int64)
parameter_table["event_id"] = event_ids
parameter_table["tel_id"] = np.full(n_events, tel_id, dtype=np.int16)
parameter_table["intensity"] = rng.uniform(90, 140, size=n_events)
parameter_table["x"] = rng.normal(0.0, 0.05, size=n_events)
parameter_table["y"] = rng.normal(0.0, 0.05, size=n_events)
parameter_table["phi"] = rng.uniform(-np.pi, np.pi, size=n_events)
parameter_table["psi"] = rng.uniform(-np.pi, np.pi, size=n_events)
parameter_table["length"] = rng.uniform(0.05, 0.15, size=n_events)
parameter_table["length_uncertainty"] = rng.uniform(0.001, 0.003, size=n_events)
parameter_table["width"] = rng.uniform(0.02, 0.08, size=n_events)
parameter_table["width_uncertainty"] = rng.uniform(0.001, 0.003, size=n_events)
parameter_table["skewness"] = rng.normal(0.0, 0.2, size=n_events)
parameter_table["kurtosis"] = rng.normal(0.0, 0.2, size=n_events)
parameter_table["time_gradient"] = rng.normal(0.0, 0.01, size=n_events)
parameter_table["intercept"] = rng.normal(0.0, 0.01, size=n_events)
parameter_table["n_pixels"] = np.full(n_events, n_pixels, dtype=np.int16)
parameter_table["n_islands"] = np.zeros(n_events, dtype=np.int16)
parameter_table["event_type"] = np.full(n_events, 32, dtype=np.int16)
parameter_table["az_tel"] = np.full(n_events, 1.0)
parameter_table["alt_tel"] = np.full(n_events, 1.2)
parameter_table["dragon_time"] = np.linspace(1_700_000_000, 1_700_000_300, n_events)

# Write to DL1 file the subarray description, image and parameter tables
subarray.to_hdf(output_path, overwrite=True)
write_table(
    image_table,
    output_path,
    "/dl1/event/telescope/image/LST_LSTCam",
    overwrite=True,
)
write_table(
    parameter_table,
    output_path,
    "/dl1/event/telescope/parameters/LST_LSTCam",
    overwrite=True,
)

In [ ]:
! ctlearn-predict-LST1 -h
#! ctlearn-predict-LST1 --help-all

In [ ]:
! mkdir ../my_predictions
input_file = "../testdata/mock_data_lst1.dl1.h5"
output_file = input_file.replace("testdata", "my_predictions").replace(".dl1.h5", ".ctlearn.dl2.h5")
! ctlearn-predict-LST1 \
    --input_url {input_file} \
    --output {output_file} \
    --config ../configs/dl1dh_example_config.json \
    --config ../configs/ctlearn_predict_model_example_config.json \
    --type_model {MODEL_DIR}/my_first_training_type/ctlearn_model.keras \
    --energy_model {MODEL_DIR}/my_first_training_energy/ctlearn_model.keras \
    --cameradirection_model {MODEL_DIR}/my_first_training_cameradirection/ctlearn_model.keras \
    --LST1PredictionTool.batch_size 5 \
    --TableQualityQuery.quality_criteria [] \
    --overwrite \
    --verbose